In [1]:
import numpy as np
import pandas as pd

# -----------------------------
# 1) Utility helpers
# -----------------------------
def clip(x, lo, hi):
    return float(np.minimum(np.maximum(x, lo), hi))

def choose_activity_level(rng):
    """
    Activity levels + realistic distribution.
    """
    levels = ["Sedentary", "Light", "Moderate", "Active"]
    probs  = [0.35, 0.35, 0.22, 0.08]
    return rng.choice(levels, p=probs)

def activity_multiplier(level):
    return {
        "Sedentary": 1.20,
        "Light": 1.375,
        "Moderate": 1.55,
        "Active": 1.725
    }[level]

def primary_goal_sampler(rng):
    goals = ["Weight loss", "Maintenance", "Glycemic control"]
    probs = [0.45, 0.40, 0.15]
    return rng.choice(goals, p=probs)

def smoking_sampler(rng):
    # 0 = no, 1 = yes
    return int(rng.choice([0, 1], p=[0.78, 0.22]))

def alcohol_sampler(rng):
    # 0 = no, 1 = yes
    return int(rng.choice([0, 1], p=[0.80, 0.20]))

def gender_sampler(rng):
    return rng.choice(["Male", "Female"], p=[0.55, 0.45])


# -----------------------------
# 2) Clinical Rule Engine
# -----------------------------
def compute_bmr(weight_kg, height_cm, age, gender):
    """
    Mifflin–St Jeor
    """
    if gender == "Male":
        return 10 * weight_kg + 6.25 * height_cm - 5 * age + 5
    else:
        return 10 * weight_kg + 6.25 * height_cm - 5 * age - 161

def compute_targets(patient):
    """
    Implements the exact clinical numeric logic described by you.

    Inputs required inside patient dict:
    age, gender, weight_kg, height_cm,
    physical_activity_level, steps_per_day,
    sleep_hours, diabetes_duration_years,
    hba1c_percent, fasting_glucose_mg_dl, postprandial_glucose_mg_dl,
    triglycerides_mg_dl, ldl_cholesterol_mg_dl, hdl_cholesterol_mg_dl,
    systolic_bp_mmHg, diastolic_bp_mmHg,
    egfr_ml_min_1_73m2,
    smoking_status, alcohol_use,
    primary_goal
    """

    age = patient["age"]
    gender = patient["gender"]
    weight_kg = patient["weight_kg"]
    height_cm = patient["height_cm"]

    activity_level = patient["physical_activity_level"]
    steps = patient["steps_per_day"]
    sleep = patient["sleep_hours"]
    diabetes_duration = patient["diabetes_duration_years"]
    hba1c = patient["hba1c_percent"]

    fasting = patient["fasting_glucose_mg_dl"]
    postprandial = patient["postprandial_glucose_mg_dl"]

    tg = patient["triglycerides_mg_dl"]
    ldl = patient["ldl_cholesterol_mg_dl"]
    hdl = patient["hdl_cholesterol_mg_dl"]

    sys_bp = patient["systolic_bp_mmHg"]
    dia_bp = patient["diastolic_bp_mmHg"]

    egfr = patient["egfr_ml_min_1_73m2"]
    smoking = patient["smoking_status"]
    alcohol = patient["alcohol_use"]
    goal = patient["primary_goal"]

    # -----------------------------
    # Step A: Calories (BMR * Activity)
    # -----------------------------
    bmr = compute_bmr(weight_kg, height_cm, age, gender)
    calories = bmr * activity_multiplier(activity_level)

    # Age penalty after 30 (−5 kcal/year after 30)
    calories -= max(age - 30, 0) * 5

    # Steps >10k => +5–10% calories
    # Use smooth gain: between 10k and 18k add up to +10%
    if steps > 10000:
        pct = clip((steps - 10000) / 8000, 0, 1)  # 0..1
        calories *= (1 + 0.10 * pct)

    # BP >140/90 => −5% calories
    if (sys_bp >= 140) or (dia_bp >= 90):
        calories *= 0.95

    # Primary goal adjustment
    if goal == "Weight loss":
        calories -= 300
    elif goal == "Glycemic control":
        # calories unchanged; carbs adjusted below
        pass

    # Safety clip (reasonable clinical)
    calories = clip(calories, 1200, 3500)

    # -----------------------------
    # Step B: Carb ratio based on HbA1c (most critical)
    # carb_ratio = 0.55 − (HbA1c − 6.5) × 0.04
    # -----------------------------
    carb_ratio = 0.55 - (hba1c - 6.5) * 0.04

    # Keep within safe ADA-ish band
    carb_ratio = clip(carb_ratio, 0.30, 0.55)

    # Diabetes duration carb reduction
    if diabetes_duration > 10:
        carb_ratio *= 0.90
    elif diabetes_duration > 5:
        carb_ratio *= 0.95

    # Fasting glucose carb reduction
    if fasting > 160:
        carb_ratio *= 0.90
    elif fasting > 130:
        carb_ratio *= 0.95

    # Postprandial carb reduction
    if postprandial > 220:
        carb_ratio *= 0.90
    elif postprandial > 180:
        carb_ratio *= 0.95

    # Sleep < 6h => −5% carbs
    if sleep < 6:
        carb_ratio *= 0.95

    # Smoking => −5% carbs
    if smoking == 1:
        carb_ratio *= 0.95

    # Alcohol => −5% carbs
    if alcohol == 1:
        carb_ratio *= 0.95

    # Primary goal: glycemic control => carbs −5%
    if goal == "Glycemic control":
        carb_ratio *= 0.95

    # Final carb ratio clip
    carb_ratio = clip(carb_ratio, 0.25, 0.55)

    # -----------------------------
    # Step C: Protein ratio (renal safety)
    # eGFR ≥90 => 20%
    # 60–89  => 18%
    # <60    => 15%
    # -----------------------------
    if egfr < 60:
        protein_ratio = 0.15
    elif egfr < 90:
        protein_ratio = 0.18
    else:
        protein_ratio = 0.20

    # slight protein preservation with age (optional tiny bump)
    # "Slight protein preservation": increase protein ratio by +0.5% after 55
    if age >= 55:
        protein_ratio += 0.005

    protein_ratio = clip(protein_ratio, 0.12, 0.25)

    # -----------------------------
    # Step D: Fat ratio remainder + lipid control penalties
    # fat_ratio = 1 − carb_ratio − protein_ratio
    # -----------------------------
    fat_ratio = 1.0 - carb_ratio - protein_ratio

    # baseline sanity: fat can't go too low
    fat_ratio = clip(fat_ratio, 0.20, 0.45)

    # TG effect (>150 => −5% fat, >250 => −10% fat)
    if tg > 250:
        fat_ratio *= 0.90
    elif tg > 150:
        fat_ratio *= 0.95

    # LDL effect (>130 => −5% fat, >160 => −10% fat)
    if ldl > 160:
        fat_ratio *= 0.90
    elif ldl > 130:
        fat_ratio *= 0.95

    # Female: slightly higher fat % (clinical tendency)
    if gender == "Female":
        fat_ratio *= 1.03

    # Alcohol: fat slightly ↑ (your rule)
    if alcohol == 1:
        fat_ratio *= 1.03

    # normalize if total > 1
    total_ratio = carb_ratio + protein_ratio + fat_ratio
    if total_ratio > 1.0:
        # renormalize only carbs+fat proportion while preserving protein safety
        remaining = 1.0 - protein_ratio
        # distribute remaining in original carb:fat proportions
        cf = carb_ratio + fat_ratio
        if cf <= 0:
            carb_ratio = remaining * 0.60
            fat_ratio  = remaining * 0.40
        else:
            carb_ratio = remaining * (carb_ratio / cf)
            fat_ratio  = remaining * (fat_ratio / cf)

    # -----------------------------
    # Step E: Convert macros to grams
    # carbs_g   = calories × carb_ratio / 4
    # protein_g = calories × protein_ratio / 4
    # fat_g     = calories × fat_ratio / 9
    # -----------------------------
    carbs_g   = calories * carb_ratio / 4.0
    protein_g = calories * protein_ratio / 4.0
    fat_g     = calories * fat_ratio / 9.0

    # extra protein scaling by weight: +0.8g per kg baseline influence
    # integrate gently: ensure at least 0.8g/kg (or renal safe lower bound 0.6g/kg)
    renal_min = 0.6 if egfr < 60 else 0.8
    protein_min = renal_min * weight_kg
    protein_g = max(protein_g, protein_min)

    # -----------------------------
    # Step F: Fiber (clinical)
    # fiber = 25 + steps/2500 + (HbA1c−6)*2 + smoking*5 + lowHDL*3
    # clip 25–50
    # -----------------------------
    fiber_g = (
        25
        + steps / 2500.0
        + (hba1c - 6.0) * 2.0
        + smoking * 5.0
        + (3.0 if hdl < 40 else 0.0)
    )

    # Sleep > 7h => +2g fiber
    if sleep > 7:
        fiber_g += 2.0

    # Activity influence fiber += steps/2500 already covers; keep that.

    fiber_g = clip(fiber_g, 25, 50)

    return {
        "daily_calories_kcal": round(calories, 1),
        "daily_carbohydrates_g": round(carbs_g, 1),
        "daily_protein_g": round(protein_g, 1),
        "daily_fat_g": round(fat_g, 1),
        "daily_fiber_g": round(fiber_g, 1),
    }


# -----------------------------
# 3) Patient Feature Sampling (realistic ranges)
# -----------------------------
def sample_patient(rng: np.random.Generator):
    gender = gender_sampler(rng)

    # Age realistic adult diabetic population
    age = int(rng.integers(18, 80))

    # Height distribution by gender
    if gender == "Male":
        height_cm = clip(rng.normal(170, 7), 155, 190)
    else:
        height_cm = clip(rng.normal(158, 6), 145, 178)

    # Weight correlated with height and age
    # BMI centered around 26-30 common in T2D
    bmi = clip(rng.normal(28, 4), 18, 42)
    weight_kg = clip((bmi * (height_cm / 100) ** 2), 45, 140)

    activity_level = choose_activity_level(rng)

    # Steps depend on activity level
    base_steps = {
        "Sedentary": rng.normal(3500, 1200),
        "Light": rng.normal(6000, 1600),
        "Moderate": rng.normal(8500, 2000),
        "Active": rng.normal(11500, 2500),
    }[activity_level]
    steps_per_day = int(clip(base_steps, 800, 20000))

    # Sleep hours
    sleep_hours = clip(rng.normal(6.8, 1.1), 4.0, 9.5)

    # Diabetes duration: correlated with age
    diabetes_duration = clip(rng.normal((age - 30) / 8, 4), 0, 30)

    # HbA1c distribution: mild to poor control
    hba1c = clip(rng.normal(7.8, 1.2), 5.6, 12.5)

    # Glucose correlated with HbA1c
    fasting_glucose = clip(rng.normal(90 + (hba1c - 5.8) * 30, 18), 70, 260)
    postprandial_glucose = clip(rng.normal(130 + (hba1c - 5.8) * 45, 25), 90, 380)

    # Lipids correlated with glycemic control + weight
    triglycerides = clip(rng.normal(130 + (hba1c - 6.5) * 18 + (bmi - 25) * 3, 35), 60, 450)
    ldl = clip(rng.normal(115 + (bmi - 25) * 1.5 + (hba1c - 6.5) * 3, 22), 50, 240)
    hdl = clip(rng.normal(48 - (bmi - 25) * 0.8, 10), 25, 85)

    # BP correlated with age & BMI
    systolic = clip(rng.normal(112 + (age - 40) * 0.6 + (bmi - 25) * 0.9, 12), 90, 190)
    diastolic = clip(rng.normal(72 + (age - 40) * 0.25 + (bmi - 25) * 0.5, 10), 55, 115)

    # Renal function: eGFR declines with age & duration
    egfr = clip(rng.normal(105 - (age - 30) * 0.6 - diabetes_duration * 0.8, 15), 20, 130)

    smoking = smoking_sampler(rng)
    alcohol = alcohol_sampler(rng)
    goal = primary_goal_sampler(rng)

    return {
        "age": int(age),
        "gender": gender,
        "height_cm": round(float(height_cm), 1),
        "weight_kg": round(float(weight_kg), 1),
        "bmi": round(float(bmi), 1),

        "physical_activity_level": activity_level,
        "steps_per_day": int(steps_per_day),
        "sleep_hours": round(float(sleep_hours), 1),

        "diabetes_duration_years": round(float(diabetes_duration), 1),
        "hba1c_percent": round(float(hba1c), 2),
        "fasting_glucose_mg_dl": round(float(fasting_glucose), 1),
        "postprandial_glucose_mg_dl": round(float(postprandial_glucose), 1),

        "triglycerides_mg_dl": round(float(triglycerides), 1),
        "ldl_cholesterol_mg_dl": round(float(ldl), 1),
        "hdl_cholesterol_mg_dl": round(float(hdl), 1),

        "systolic_bp_mmHg": round(float(systolic), 1),
        "diastolic_bp_mmHg": round(float(diastolic), 1),

        "egfr_ml_min_1_73m2": round(float(egfr), 1),

        "smoking_status": int(smoking),
        "alcohol_use": int(alcohol),
        "primary_goal": goal
    }


# -----------------------------
# 4) Dataset Generator
# -----------------------------
def generate_dataset(n_rows=5000, seed=42, out_csv="clinical_synthetic_nutrition.csv"):
    rng = np.random.default_rng(seed)
    rows = []

    for _ in range(n_rows):
        patient = sample_patient(rng)
        targets = compute_targets(patient)
        row = {**patient, **targets}
        rows.append(row)

    df = pd.DataFrame(rows)

    # Nice ordering
    feature_cols = [
        "age", "gender", "height_cm", "weight_kg", "bmi",
        "physical_activity_level", "steps_per_day", "sleep_hours",
        "diabetes_duration_years", "hba1c_percent",
        "fasting_glucose_mg_dl", "postprandial_glucose_mg_dl",
        "triglycerides_mg_dl", "ldl_cholesterol_mg_dl", "hdl_cholesterol_mg_dl",
        "systolic_bp_mmHg", "diastolic_bp_mmHg",
        "egfr_ml_min_1_73m2",
        "smoking_status", "alcohol_use", "primary_goal"
    ]

    target_cols = [
        "daily_calories_kcal",
        "daily_carbohydrates_g",
        "daily_protein_g",
        "daily_fat_g",
        "daily_fiber_g"
    ]

    df = df[feature_cols + target_cols]

    df.to_csv(out_csv, index=False)
    print(f"✅ Generated dataset: {out_csv}")
    print("Shape:", df.shape)
    print(df.head(3))

    return df


# -----------------------------
# 5) Run directly
# -----------------------------
if __name__ == "__main__":
    generate_dataset(n_rows=10000, seed=7, out_csv="clinical_synthetic_nutrition.csv")


✅ Generated dataset: clinical_synthetic_nutrition.csv
Shape: (10000, 26)
   age  gender  height_cm  weight_kg   bmi physical_activity_level  \
0   60  Female      156.4       59.7  24.4               Sedentary   
1   73    Male      171.1       79.8  27.3                   Light   
2   33    Male      170.8       74.2  25.4                   Light   

   steps_per_day  sleep_hours  diabetes_duration_years  hba1c_percent  ...  \
0           2310          6.1                      5.7           8.23  ...   
1           5922          6.3                      1.5           6.83  ...   
2           4081          6.6                      3.1           7.72  ...   

   diastolic_bp_mmHg  egfr_ml_min_1_73m2  smoking_status  alcohol_use  \
0               57.7                63.1               0            0   
1               82.5                79.0               1            0   
2               58.6                92.0               0            0   

       primary_goal  daily_calories_kcal